In [1]:
from __future__ import annotations

import json
import math
import random
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import librosa
from torchvision.models import alexnet, AlexNet_Weights

print("EXPERIMENT8B — VOC-ALS dysarthria (paper protocol + paper classifier Hypernetwork)")

# ====================== PATHS ======================
REPO_ROOT = Path("/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta")
VOC_ROOT = REPO_ROOT / "Dataset" / "RUSSIAN"
VOC_META_XLSX = VOC_ROOT / "VOC-ALS.xlsx"
VOC_RHYTHM_PA = VOC_ROOT / "rhythmPA"

RESULTS_DIR = REPO_ROOT / "Results" / "AL_MODELS" / "exp8b_vocals_hypernetwork_paper_repro"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CACHE_NPZ = RESULTS_DIR / "cache_rhythmPA_logmel_delta224.npz"
FOLD_CSV = RESULTS_DIR / "fold_metrics.csv"
SUMMARY_JSON = RESULTS_DIR / "summary.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("VOC_ROOT:", VOC_ROOT, "| exists:", VOC_ROOT.exists())
print("VOC_META_XLSX:", VOC_META_XLSX, "| exists:", VOC_META_XLSX.exists())
print("VOC_RHYTHM_PA:", VOC_RHYTHM_PA, "| exists:", VOC_RHYTHM_PA.exists())
print("RESULTS_DIR:", RESULTS_DIR)
print("device:", DEVICE)

# ====================== PAPER SETTINGS (as stated) ======================
# Input: rhythmPA, log-Mel + delta + delta-delta
N_MELS = 256
HOP_LENGTH = 512

# Librosa defaults (paper doesn't specify these; keep defaults to be closest to typical usage)
SR = 22050
N_FFT = 2048
POWER = 2.0

# Resize to AlexNet input
IMG_SIZE = 224

# CV protocol
N_SPLITS = 5
N_REPEATS = 4
BASE_SEED = 42

# Training
NUM_EPOCHS = 30
LR = 1e-5
BATCH_SIZE = 16
WEIGHT_DECAY = 0.0

# Hypernetwork paper params
# If True, sample one condition vector once and reuse it for all forward passes (more stable).
FIXED_CONDITION_VECTOR = True
# If >1 and FIXED_CONDITION_VECTOR=False, average predictions over MC samples at eval time.
EVAL_MC_SAMPLES = 1

COND_DIM = 128
HNET_HIDDEN = 512
ALEX_OUT_DIM = 768
N_CLASSES = 2

# AlexNet: pretrained, modified
ALEXNET_PRETRAINED = True
FINE_TUNE_ALEXNET = True

# ====================== XLSX READ (openpyxl fallback) ======================
def _xlsx_col_to_index(col: str) -> int:
    n = 0
    for ch in col:
        if not ('A' <= ch <= 'Z'):
            break
        n = n * 26 + (ord(ch) - ord('A') + 1)
    return n


def read_xlsx_basic(path: Path) -> pd.DataFrame:
    import zipfile
    from xml.etree import ElementTree as ET

    with zipfile.ZipFile(path, 'r') as z:
        shared = []
        if 'xl/sharedStrings.xml' in z.namelist():
            root = ET.fromstring(z.read('xl/sharedStrings.xml'))
            ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
            for si in root.findall('m:si', ns):
                texts = [t.text or '' for t in si.findall('.//m:t', ns)]
                shared.append(''.join(texts))

        sheet_path = None
        try:
            wb = ET.fromstring(z.read('xl/workbook.xml'))
            ns = {
                'm': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
                'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
            }
            sheet = wb.find('m:sheets/m:sheet', ns)
            rid = sheet.attrib.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
            rels = ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))
            nsr = {'r': 'http://schemas.openxmlformats.org/package/2006/relationships'}
            target = None
            for rel in rels.findall('r:Relationship', nsr):
                if rel.attrib.get('Id') == rid:
                    target = rel.attrib.get('Target')
                    break
            if target:
                target = target.lstrip('/')
                if not target.startswith('xl/'):
                    target = 'xl/' + target
                if target in z.namelist():
                    sheet_path = target
        except Exception:
            sheet_path = None

        if sheet_path is None:
            sheet_path = 'xl/worksheets/sheet1.xml'
        if sheet_path not in z.namelist():
            raise FileNotFoundError(f'Could not locate worksheet xml in xlsx (tried {sheet_path})')

        sh = ET.fromstring(z.read(sheet_path))
        ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

        cells = {}
        max_r = 0
        max_c = 0

        for row in sh.findall('.//m:sheetData/m:row', ns):
            for c in row.findall('m:c', ns):
                ref = c.attrib.get('r')
                if not ref:
                    continue
                m = re.match(r'([A-Z]+)(\d+)', ref)
                if not m:
                    continue
                col_letters, row_num = m.group(1), int(m.group(2))
                col_num = _xlsx_col_to_index(col_letters)

                t = c.attrib.get('t')
                if t == 'inlineStr':
                    is_el = c.find('m:is', ns)
                    texts = []
                    if is_el is not None:
                        texts = [t2.text or '' for t2 in is_el.findall('.//m:t', ns)]
                    v = ''.join(texts)
                else:
                    v_el = c.find('m:v', ns)
                    raw = None if v_el is None else v_el.text
                    if raw is None:
                        v = None
                    elif t == 's':
                        try:
                            v = shared[int(raw)]
                        except Exception:
                            v = raw
                    else:
                        try:
                            v = float(raw) if '.' in raw else int(raw)
                        except Exception:
                            v = raw

                cells[(row_num, col_num)] = v
                max_r = max(max_r, row_num)
                max_c = max(max_c, col_num)

        data = [[None for _ in range(max_c)] for _ in range(max_r)]
        for (r, c), v in cells.items():
            data[r - 1][c - 1] = v

        return pd.DataFrame(data)


def load_vocals_xlsx(path: Path) -> pd.DataFrame:
    try:
        df0 = pd.read_excel(path, header=None)
    except Exception as e:
        if 'openpyxl' in str(e).lower():
            df0 = read_xlsx_basic(path)
        else:
            raise

    header_row = None
    for i in range(min(10, len(df0))):
        if (df0.iloc[i].astype(str).str.strip() == 'ID').any():
            header_row = i
            break
    if header_row is None:
        raise ValueError('Could not find header row containing ID')

    header = df0.iloc[header_row].astype(str).tolist()
    df = df0.iloc[header_row + 1 :].copy()
    df.columns = header
    return df.reset_index(drop=True)


def subject_id_from_filename(name: str) -> str:
    m = re.match(r'^((?:CT|PZ)\d+)_', name, flags=re.IGNORECASE)
    if not m:
        raise ValueError(f'Could not parse subject ID from filename: {name}')
    return m.group(1).upper()


# ====================== BUILD DATASET (ALS-only dysarthria labels) ======================
df_meta = load_vocals_xlsx(VOC_META_XLSX)
need_cols = {'ID', 'Category', 'ALSFRS-R_SpeechSubscore'}
missing = [c for c in need_cols if c not in df_meta.columns]
if missing:
    raise ValueError(f'Missing required columns in VOC-ALS.xlsx: {missing}')

df_als = df_meta[df_meta['Category'].astype(str).str.strip().eq('ALS')].copy()
df_als['ID'] = df_als['ID'].astype(str).str.strip().str.upper()
df_als['speech'] = pd.to_numeric(df_als['ALSFRS-R_SpeechSubscore'], errors='coerce')
df_als = df_als.dropna(subset=['speech']).copy()
df_als['speech'] = df_als['speech'].astype(int)

# dysarthric vs non-dysarthric (speech<4 vs speech==4)
df_als['y'] = (df_als['speech'] < 4).astype(int)

y_by_id = {r['ID']: int(r['y']) for _, r in df_als.iterrows()}

wav_paths = sorted([p for p in VOC_RHYTHM_PA.glob('*.wav') if p.is_file()])
rows = []
for wp in wav_paths:
    sid = subject_id_from_filename(wp.name)
    if sid not in y_by_id:
        continue
    rows.append({'subject_id': sid, 'wav_path': str(wp), 'y': y_by_id[sid]})

df = pd.DataFrame(rows)
print('ALS subjects (metadata):', len(df_als))
print('Matched rhythmPA files:', len(df), '| unique subjects:', df['subject_id'].nunique())
print('Label counts (0=non-dys,1=dys):', df['y'].value_counts().sort_index().to_dict())

if df['subject_id'].nunique() != len(df):
    print('[WARN] Multiple rhythmPA files per subject detected; this notebook still does subject-wise CV to avoid leakage.')


# ====================== FEATURE EXTRACTION (paper-aligned) ======================
def wav_to_logmel_delta3ch(path: str) -> np.ndarray:
    # librosa.load default sr=22050 in many papers; use SR for reproducibility
    y, _sr = librosa.load(path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        power=POWER,
    )
    # Use dB scale (common for log-mel)
    logS = librosa.power_to_db(S, ref=np.max).astype(np.float32)
    d1 = librosa.feature.delta(logS).astype(np.float32)
    d2 = librosa.feature.delta(logS, order=2).astype(np.float32)
    x = np.stack([logS, d1, d2], axis=0)  # (3, 256, T)

    xt = torch.from_numpy(x)[None, ...]
    xt = F.interpolate(xt, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
    return xt.squeeze(0).numpy().astype(np.float32)


def build_or_load_cache() -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    if CACHE_NPZ.exists():
        c = np.load(CACHE_NPZ, allow_pickle=True)
        return c['X_img'].astype(np.float32), c['y'].astype(np.int64), c['subject_ids'].astype(object)

    X_img = []
    y_list = []
    sids = []
    for r in tqdm(df.itertuples(index=False), total=len(df), desc='cache logmel'):
        X_img.append(wav_to_logmel_delta3ch(r.wav_path))
        y_list.append(int(r.y))
        sids.append(str(r.subject_id))

    X_img = np.stack(X_img, axis=0)  # (N,3,224,224)
    y_arr = np.asarray(y_list, dtype=np.int64)
    sid_arr = np.asarray(sids, dtype=object)
    np.savez_compressed(CACHE_NPZ, X_img=X_img, y=y_arr, subject_ids=sid_arr)
    print('Saved cache:', CACHE_NPZ)
    return X_img, y_arr, sid_arr


X_img_all, y_all, subject_ids = build_or_load_cache()
print('X_img_all:', X_img_all.shape, '| y_all:', y_all.shape, '| subjects:', len(np.unique(subject_ids)))


# ====================== MODEL (paper classifier) ======================
class ModifiedAlexNet(nn.Module):
    def __init__(self, out_dim: int = 768, pretrained: bool = True):
        super().__init__()
        weights = AlexNet_Weights.DEFAULT if pretrained else None
        try:
            base = alexnet(weights=weights)
        except Exception as e:
            raise RuntimeError(
                'Failed to load AlexNet. If pretrained weights are not cached locally, disable pretrained or download weights.'
            ) from e

        # Replace last classifier layer with out_dim (this yields D=out_dim)
        base.classifier[6] = nn.Linear(4096, int(out_dim))
        self.base = base

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x)


class HyperNetwork(nn.Module):
    def __init__(self, cond_dim: int = 128, hidden_dim: int = 512, in_dim: int = 768, n_classes: int = 2):
        super().__init__()
        self.in_dim = int(in_dim)
        self.n_classes = int(n_classes)
        self.backbone = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.ReLU(),
        )
        # Paper says output layer 1536 units (768x2). We add a separate bias head to match their text.
        self.fc_w = nn.Linear(hidden_dim, self.in_dim * self.n_classes)
        self.fc_b = nn.Linear(hidden_dim, self.n_classes)

    def forward(self, c: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.backbone(c)
        w = self.fc_w(h)  # (B, in_dim*n_classes)
        b = self.fc_b(h)  # (B, n_classes)
        w = w.view(-1, self.in_dim, self.n_classes)
        return w, b


class PaperHyperNetClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.alex = ModifiedAlexNet(out_dim=ALEX_OUT_DIM, pretrained=ALEXNET_PRETRAINED)
        self.hnet = HyperNetwork(cond_dim=COND_DIM, hidden_dim=HNET_HIDDEN, in_dim=ALEX_OUT_DIM, n_classes=N_CLASSES)
        self.register_buffer("c_fixed", torch.randn(1, COND_DIM))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,3,224,224)
        feat = self.alex(x)  # (B,768)
        # condition vector ~ N(0,1)
        if FIXED_CONDITION_VECTOR:
            c = self.c_fixed.expand(x.size(0), -1)
        else:
            c = torch.randn((x.size(0), COND_DIM), device=x.device)
        W, b = self.hnet(c)  # W: (B,768,2) b: (B,2)
        # target net: y = X @ W + b, per-sample
        y = torch.bmm(feat.unsqueeze(1), W).squeeze(1) + b
        return y


def set_trainable(model: nn.Module, train_alexnet: bool):
    for p in model.alex.parameters():
        p.requires_grad = bool(train_alexnet)


# ====================== DATASET ======================
class VocALSImgDataset(Dataset):
    def __init__(self, X_img: np.ndarray, y: np.ndarray):
        self.X = X_img
        self.y = y.astype(np.int64)

    def __len__(self):
        return int(len(self.X))

    def __getitem__(self, idx: int):
        x = torch.from_numpy(self.X[idx]).float()
        y = int(self.y[idx])
        return x, y


def normalize_img_batch(x: torch.Tensor) -> torch.Tensor:
    # Use ImageNet normalization (torchvision convention)
    mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(1, 3, 1, 1)
    return (x - mean) / std


# ====================== TRAIN/EVAL ======================
def train_one_run(model: nn.Module, train_loader: DataLoader, seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()

    for _epoch in range(int(NUM_EPOCHS)):
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            xb = normalize_img_batch(xb)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()


@torch.no_grad()
def eval_one_run(model: nn.Module, test_loader: DataLoader) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    probs = []
    preds = []
    trues = []
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        xb = normalize_img_batch(xb)
        if (not FIXED_CONDITION_VECTOR) and int(EVAL_MC_SAMPLES) > 1:
            ps = []
            for _ in range(int(EVAL_MC_SAMPLES)):
                logits = model(xb)
                ps.append(torch.softmax(logits, dim=1))
            p_t = torch.stack(ps, dim=0).mean(dim=0)
        else:
            logits = model(xb)
            p_t = torch.softmax(logits, dim=1)
        p = p_t.detach().cpu().numpy()
        yhat = np.argmax(p, axis=1)
        probs.append(p)
        preds.append(yhat)
        trues.append(yb.numpy())
    return np.vstack(probs), np.concatenate(preds), np.concatenate(trues)


def specificity(cm: np.ndarray) -> float:
    tn, fp, fn, tp = cm.ravel()
    return float(tn / max(1, (tn + fp)))


# ====================== CV (subject-wise) ======================
uniq_subjects = np.unique(subject_ids.astype(str))
subj_y = []
for sid in uniq_subjects:
    ys = y_all[subject_ids.astype(str) == sid]
    vals, cnts = np.unique(ys, return_counts=True)
    subj_y.append(int(vals[np.argmax(cnts)]))
subj_y = np.asarray(subj_y, dtype=int)

fold_rows = []
rep_seeds = [BASE_SEED + r for r in range(N_REPEATS)]

for rep_i, rep_seed in enumerate(rep_seeds):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=int(rep_seed))
    for fold_i, (tr_s, te_s) in enumerate(skf.split(uniq_subjects, subj_y)):
        tr_subjects = uniq_subjects[tr_s]
        te_subjects = uniq_subjects[te_s]

        tr_idx = np.where(np.isin(subject_ids.astype(str), tr_subjects))[0]
        te_idx = np.where(np.isin(subject_ids.astype(str), te_subjects))[0]

        fold_seed = int(rep_seed * 100 + fold_i)

        train_ds = VocALSImgDataset(X_img_all[tr_idx], y_all[tr_idx])
        test_ds = VocALSImgDataset(X_img_all[te_idx], y_all[te_idx])

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

        model = PaperHyperNetClassifier().to(DEVICE)
        set_trainable(model, train_alexnet=FINE_TUNE_ALEXNET)

        train_one_run(model, train_loader, seed=fold_seed)
        proba, y_pred, y_true = eval_one_run(model, test_loader)

        acc = float(accuracy_score(y_true, y_pred))
        prec = float(precision_score(y_true, y_pred, pos_label=1, zero_division=0))
        rec = float(recall_score(y_true, y_pred, pos_label=1, zero_division=0))
        f1 = float(f1_score(y_true, y_pred, pos_label=1, zero_division=0))
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        spec = float(specificity(cm))
        try:
            auc = float(roc_auc_score(y_true, proba[:, 1]))
        except ValueError:
            auc = float('nan')

        fold_rows.append(
            {
                'repeat': int(rep_i),
                'fold': int(fold_i),
                'seed': int(fold_seed),
                'n_train_subjects': int(len(tr_subjects)),
                'n_test_subjects': int(len(te_subjects)),
                'n_train': int(len(tr_idx)),
                'n_test': int(len(te_idx)),
                'acc': acc,
                'f1': f1,
                'auc': auc,
                'precision': prec,
                'recall': rec,
                'specificity': spec,
                'tn': int(cm[0, 0]),
                'fp': int(cm[0, 1]),
                'fn': int(cm[1, 0]),
                'tp': int(cm[1, 1]),
            }
        )

        print(f"rep={rep_i} fold={fold_i} acc={acc:.4f} f1={f1:.4f} auc={auc:.4f} prec={prec:.4f} rec={rec:.4f} spec={spec:.4f}")


df_folds = pd.DataFrame(fold_rows)
df_folds.to_csv(FOLD_CSV, index=False)
print('Saved:', FOLD_CSV)

summary = {
    'paper': 'Recognition of Dysarthria in ALS patients using Hypernetworks (EUSIPCO 2025)',
    'task': 'VOC-ALS dysarthria recognition within ALS (speech_subscore <4 vs =4)',
    'input': 'rhythmPA',
    'cv': {'n_splits': int(N_SPLITS), 'n_repeats': int(N_REPEATS), 'base_seed': int(BASE_SEED)},
    'features': {'sr': int(SR), 'n_mels': int(N_MELS), 'hop_length': int(HOP_LENGTH), 'n_fft': int(N_FFT), 'img_size': int(IMG_SIZE)},
    'model': {
        'alexnet_pretrained': bool(ALEXNET_PRETRAINED),
        'fine_tune_alexnet': bool(FINE_TUNE_ALEXNET),
        'alex_out_dim': int(ALEX_OUT_DIM),
        'cond_dim': int(COND_DIM),
        'hyper_hidden': int(HNET_HIDDEN),
        'epochs': int(NUM_EPOCHS),
        'lr': float(LR),
        'batch_size': int(BATCH_SIZE),
    },
    'data': {
        'n_subjects': int(len(uniq_subjects)),
        'label_counts_subjects': {
            'non_dys': int((subj_y == 0).sum()),
            'dys': int((subj_y == 1).sum()),
        },
    },
    'results': {
        'mean': df_folds[['acc','f1','auc','precision','recall','specificity']].mean().to_dict(),
        'std': df_folds[['acc','f1','auc','precision','recall','specificity']].std(ddof=0).to_dict(),
    },
}

with open(SUMMARY_JSON, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('Saved:', SUMMARY_JSON)

print('\n=== FINAL (mean ± std over folds×repeats) ===')
for k in ['acc','f1','auc','precision','recall','specificity']:
    mu = summary['results']['mean'][k]
    sd = summary['results']['std'][k]
    print(f"{k:12s}: {mu:.4f} ± {sd:.4f}")


EXPERIMENT8B — VOC-ALS dysarthria (paper protocol + paper classifier Hypernetwork)
VOC_ROOT: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/RUSSIAN | exists: True
VOC_META_XLSX: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/RUSSIAN/VOC-ALS.xlsx | exists: True
VOC_RHYTHM_PA: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Dataset/RUSSIAN/rhythmPA | exists: True
RESULTS_DIR: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/exp8b_vocals_hypernetwork_paper_repro
device: cuda
ALS subjects (metadata): 102
Matched rhythmPA files: 102 | unique subjects: 102
Label counts (0=non-dys,1=dys): {0: 53, 1: 49}
X_img_all: (102, 3, 224, 224) | y_all: (102,) | subjects: 102
rep=0 fold=0 acc=0.6190 f1=0.6000 auc=0.6500 prec=0.6000 rec=0.6000 spec=0.6364
rep=0 fold=1 acc=0.7143 f1=0.7500 auc=0.7636 prec=0.6429 rec=0.9000 spec=0.5455
rep=0 fold=2 acc=0.7500 f1=0.7826 auc=0.8600 prec=0.6923 rec=0.9000 spec=0.6000
rep=0 fold=3 acc=0.5000 f1=0.5000 auc=0.5450 prec=0.5000 rec=0.5000 spec=0.5000